In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/processed/cleaned_telco.csv")

cox = joblib.load("../models/cox_model.pkl")

df.head()

In [ ]:
active = df[df["Churn"] == 0].copy()

print("Active customers:", len(active))

In [ ]:
features = active[
    ["MonthlyCharges", "SeniorCitizen",
     "Contract", "InternetService"]
].copy()

features = pd.get_dummies(
    features,
    columns=["Contract", "InternetService"],
    drop_first=True,
    dtype=int
)

features = features.reindex(
    columns=cox.params_.index,
    fill_value=0
)

In [ ]:
max_tenure = int(cox.baseline_survival_.index.max())

remaining_months = []

for i in range(len(active)):
    customer = features.iloc[[i]]
    tenure = active["tenure"].iloc[i]

    horizon = min(12, max(0, max_tenure - int(tenure)))

    if horizon == 0:
        remaining_months.append(0)
        continue

    curve = cox.predict_survival_function(
        customer,
        times=np.arange(1, horizon + 1),
        conditional_after=[tenure]
    )

    remaining_months.append(curve.iloc[:, 0].sum())

active["ExpectedRemainingMonths"] = remaining_months

In [ ]:
profit_margin = 0.40

active["CLV"] = (
    active["MonthlyCharges"]
    * active["ExpectedRemainingMonths"]
    * profit_margin
).round(2)

active[
    ["customerID", "MonthlyCharges",
     "ExpectedRemainingMonths", "CLV"]
].head(10)

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(active["CLV"], bins=30, kde=True)

plt.title("Customer Lifetime Value Distribution")
plt.xlabel("Estimated CLV")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
active.to_csv(
    "../data/processed/customer_clv.csv",
    index=False
)

print("CLV dataset saved successfully!")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
cluster_features = active[
    ["tenure", "MonthlyCharges", "CLV"]
].copy()

cluster_features.head()

In [ ]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(cluster_features)

In [ ]:
inertia = []

for k in range(1, 11):
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    km.fit(scaled_features)
    inertia.append(km.inertia_)

plt.plot(range(1, 11), inertia, marker="o")

plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

active["Segment"] = kmeans.fit_predict(scaled_features)

active["Segment"].value_counts()

In [ ]:
score = silhouette_score(
    scaled_features,
    active["Segment"]
)

print("Silhouette Score:", score)

In [ ]:
segment_summary = active.groupby("Segment")[
    ["tenure", "MonthlyCharges", "CLV"]
].mean().round(2)

segment_summary

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=active,
    x="tenure",
    y="CLV",
    hue="Segment",
    palette="Set1"
)

plt.title("Customer Segmentation")
plt.show()

In [ ]:
joblib.dump(kmeans, "../models/kmeans_model.pkl")

joblib.dump(scaler, "../models/cluster_scaler.pkl")

active.to_csv(
    "../data/processed/customer_segments.csv",
    index=False
)

print("Customer segmentation completed!")

In [ ]:
segment_names = {
    0: "New Budget Customers",
    1: "High-Value Customers",
    2: "Long-Term Premium Customers",
    3: "Loyal Budget Customers"
}

active["SegmentName"] = active["Segment"].map(segment_names)

active[["customerID", "Segment", "SegmentName", "CLV"]].head()

In [ ]:
active.to_csv(
    "../data/processed/customer_segments.csv",
    index=False
)

print("Customer segment names saved successfully!")